#Mini Trabalho DAA

##Ano Letivo 25/26

###Trabalho Realizado por:


*   Daniel Silva, nº 129859
*   Francisco Silva, nº 129868
*   Rodrigo Cruz, nº 129829

#Análise de Redes

A análise de redes constitui uma área central da teoria dos grafos, com aplicações que vão desde redes sociais e infraestruturas até sistemas biológicos. Neste trabalho, no âmbito da unidade curricular de Desenho e Análise de Algoritmos (DAA), aplicamos estas técnicas ao estudo de universos ficcionais — concretamente, às redes de co-ocorrências de personagens da saga A Song of Ice and Fire (Game of Thrones) e do universo Marvel Comics.
O objetivo principal foi implementar, de raiz em Python, um conjunto de métricas estatísticas para análise de grafos não orientados, organizadas numa API denominada CentralityAnalyzer. Esta implementação foi desenvolvida sem recurso a bibliotecas externas de análise de grafos (como networkx ou igraph), partindo da estrutura de dados Graph desenvolvida nas aulas práticas.
O trabalho está organizado nas seguintes componentes:

* Análise Estrutural — implementação de BFS, cálculo de componentes conexas, distribuição de graus e diâmetro da maior componente conexa;

* Métricas de Centralidade — implementação e análise de Degree Centrality, Closeness Centrality, Eigenvector Centrality (via Power Iteration) e Betweenness Centrality (algoritmo de Brandes);

* Análise de Escalabilidade — estudo empírico do comportamento temporal dos algoritmos implementados nos quatro datasets fornecidos, de dimensão variável (desde 187 vértices até 6421 vértices).

Para cada método implementado, é apresentada a análise de complexidade temporal, validação dos resultados e interpretação no contexto narrativo dos universos ficcionais analisados. Os datasets utilizados incluem got_book1.csv, got_full.csv, marvel_small.csv e marvel_full.csv, permitindo comparar o comportamento dos algoritmos em redes de diferentes escalas.




##   Bibliotecas usadas







In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import time
import random
from collections import deque
import matplotlib.ticker as ticker
import csv

## TDA Graph

Código adaptado das aulas da semana 7.


In [ ]:
class Vertex:
    def __init__(self, vertex_id):
        self._vertex_id = vertex_id

    def __hash__(self):
        return hash(self._vertex_id)

    def __str__(self):
        return 'v{0}'.format(self._vertex_id)

    def __eq__(self, vertex):
        return self._vertex_id == vertex._vertex_id

    def vertex_id(self):
        return self._vertex_id


class Edge:
    def __init__(self, vertex_1, vertex_2, weight):
        self._vertex_1 = vertex_1
        self._vertex_2 = vertex_2
        self._weight = weight

    def __hash__(self):
        return hash((self._vertex_1, self._vertex_2))

    def __str__(self):
        return 'e({0},{1})w={2}'.format(self._vertex_1, self._vertex_2, self._weight)

    def endpoints(self):
        return (self._vertex_1, self._vertex_2)

    def cost(self):
        return self._weight

    def opposite(self, vertex):
        if vertex == self._vertex_1:
            return self._vertex_2
        elif vertex == self._vertex_2:
            return self._vertex_1
        else:
            return None


class Graph:
    def __init__(self):
        self._adjancencies = {}
        self._vertices = {}
        self._n = 0
        self._m = 0

    def __str__(self):
        if self._n == 0:
            return "DAA-Graph: <empty>\n"
        ret = "DAA-Graph:\n"
        for vertex in self._adjancencies.keys():
            ret += str(vertex) + ": "
            for edge in self.incident_edges(vertex.vertex_id()):
                ret += str(edge) + "; "
            ret += "\n"
        return ret

    def order(self):
        return self._n

    def size(self):
        return self._m

    def has_vertex(self, vertex_id):
        return vertex_id in self._vertices

    def has_edge(self, u_id, v_id):
        if not self.has_vertex(u_id) or not self.has_vertex(v_id):
            return False
        vertex_u = self._vertices[u_id]
        vertex_v = self._vertices[v_id]
        return vertex_v in self._adjancencies[vertex_u]

    def insert_vertex(self, vertex_id):
        if not self.has_vertex(vertex_id):
            vertex = Vertex(vertex_id)
            self._vertices[vertex_id] = vertex
            self._adjancencies[vertex] = {}
            self._n += 1

    def insert_edge(self, u_id, v_id, weight=0):
        if not self.has_vertex(u_id):
            self.insert_vertex(u_id)
        if not self.has_vertex(v_id):
            self.insert_vertex(v_id)
        if not self.has_edge(u_id, v_id):
            self._m += 1
        vertex_u = self._vertices[u_id]
        vertex_v = self._vertices[v_id]
        e = Edge(vertex_u, vertex_v, weight)
        self._adjancencies[vertex_u][vertex_v] = e
        self._adjancencies[vertex_v][vertex_u] = e

    def degree(self, vertex_id):
        return len(self._adjancencies[self._vertices[vertex_id]])

    def get_vertex(self, vertex_id):
        return None if not self.has_vertex(vertex_id) else self._vertices[vertex_id]

    def get_edge(self, u_id, v_id):
        if not self.has_edge(u_id, v_id):
            return None
        vertex_u = self._vertices[u_id]
        vertex_v = self._vertices[v_id]
        return self._adjancencies[vertex_u][vertex_v]

    def vertices(self):
        return self._vertices.values()

    def edges(self):
        seen = {}
        for adj_map in self._adjancencies.values():
            for edge in adj_map.values():
                if edge not in seen:
                    yield edge
                seen[edge] = True

    def incident_edges(self, vertex_id):
        vertex = self._vertices[vertex_id]
        for edge in self._adjancencies[vertex].values():
            yield edge

    def remove_vertex(self, vertex_id):
        if self.has_vertex(vertex_id):
            lst_copied = list(self.incident_edges(vertex_id))
            for edge in lst_copied:
                x, y = edge.endpoints()
                self.remove_edge(x.vertex_id(), y.vertex_id())
            del self._adjancencies[self._vertices[vertex_id]]
            del self._vertices[vertex_id]
            self._n -= 1

    def remove_edge(self, u_id, v_id):
        if self.has_edge(u_id, v_id):
            vertex_u = self._vertices[u_id]
            vertex_v = self._vertices[v_id]
            del self._adjancencies[vertex_u][vertex_v]
            if vertex_u != vertex_v:
                del self._adjancencies[vertex_v][vertex_u]
            self._m -= 1

    @staticmethod
    def from_csv(filepath):
        g = Graph()
        with open(filepath, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                src = row['Source'].strip()
                tgt = row['Target'].strip()
                w   = float(row.get('weight', 1))
                g.insert_edge(src, tgt, w)
        return g

##1. API CentralityAnalyzer

In [ ]:
#API para análise de centralidade de grafos ficcionais.
class CentralityAnalyzer:

    # Recebe um objecto Graph já construído (por Graph.from_csv).
    def __init__(self, graph: Graph):
        self._graph = graph
        self._n = graph.num_vertices()
        self._m = graph.num_edges()


    def bfs(self,source):
        """
        Travessia em largura a partir de `source`.
        Devolve (dist, pred):
          - dist: dict {v -> distância BFS de source a v}
          - pred: dict {v -> predecessor na BFS tree}
        """
        dist = {source: 0}
        pred = {source: None}
        queue = deque([source])
        while queue:
            u = queue.popleft()
            for (v, _) in self._graph.neighbors(u):
                if v not in dist:
                    dist[v] = dist[u] + 1
                    pred[v] = u
                    queue.append(v)
        return dist, pred

        def num_components(self):
        visited = set()
        count = 0
        for v in self._graph.vertices():
            v_id = v.vertex_id()
            if v_id not in visited:
                count += 1
                dist, _ = self.bfs(v_id)
                visited.update(dist.keys())
        return count

    def largest_component(self):
        """Retorna um novo objeto Graph que é a maior componente conexa."""
        visited = set()
        largest_nodes = []
        for v in self._graph.vertices():
            v_id = v.vertex_id()
            if v_id not in visited:
                dist, _ = self.bfs(v_id)
                nodes = list(dist.keys())
                if len(nodes) > len(largest_nodes):
                    largest_nodes = nodes
                visited.update(nodes)

        new_g = Graph()
        # Adicionar vértices e arestas à nova componente
        for u_id in largest_nodes:
            new_g.insert_vertex(u_id)
        for u_id in largest_nodes:
            for edge in self._graph.incident_edges(u_id):
                v1, v2 = edge.endpoints()
                v1_id, v2_id = v1.vertex_id(), v2.vertex_id()
                if v2_id in largest_nodes:
                    new_g.insert_edge(v1_id, v2_id, edge.cost())
        return new_g

    def degree_distribution(self):
        distrib = {}
        for v in self._graph.vertices():
            deg = self._graph.degree(v.vertex_id())
            distrib[deg] = distrib.get(deg, 0) + 1
        return distrib

    def diameter(self):
        """Cálculo do diâmetro (na componente atual)."""
        max_dist = 0
        # Otimização: se o grafo for grande, este processo é O(V*(V+E))
        for v in self._graph.vertices():
            dist, _ = self.bfs(v.vertex_id())
            current_max = max(dist.values()) if dist else 0
            if current_max > max_dist:
                max_dist = current_max
        return max_dist

    # --- 2. Métricas de Centralidade ---

    def degree_centrality(self):
        res = {}
        for v in self._graph.vertices():
            v_id = v.vertex_id()
            res[v_id] = self._graph.degree(v_id) / (self._n - 1)
        return res

    def closeness_centrality(self):
        res = {}
        for v in self._graph.vertices():
            v_id = v.vertex_id()
            dist, _ = self.bfs(v_id)
            sum_dist = sum(dist.values())
            if sum_dist > 0 and len(dist) > 1:
                # Fórmula normalizada para a componente
                res[v_id] = (len(dist) - 1) / sum_dist
            else:
                res[v_id] = 0.0
        return res

    def eigenvector_centrality(self, max_iter=100, tol=1e-6):
        """Implementação via Power Iteration."""
        # Inicializar vetor
        nodes = [v.vertex_id() for v in self._graph.vertices()]
        x = {node: 1.0 / self._n for node in nodes}

        for _ in range(max_iter):
            x_last = x.copy()
            # Multiplicação pela matriz de adjacência
            for u in nodes:
                total = 0
                # Vizinhos do vértice u
                u_vertex = self._graph.get_vertex(u)
                for v_vertex in self._graph._adjancencies[u_vertex]:
                    total += x_last[v_vertex.vertex_id()]
                x[u] = total

            # Normalização (Norma L2)
            norm = math.sqrt(sum(v**2 for v in x.values()))
            if norm == 0: break
            for u in x: x[u] /= norm

            # Verificar convergência
            error = sum(abs(x[u] - x_last[u]) for u in nodes)
            if error < tol:
                break
        return x

    def betweenness_centrality(self):
        """Algoritmo de Brandes O(VE)."""
        betweenness = {v.vertex_id(): 0.0 for v in self._graph.vertices()}

        for s in [v.vertex_id() for v in self._graph.vertices()]:
            # Passo 1: BFS para distâncias e caminhos mínimos
            S = []
            P = {v.vertex_id(): [] for v in self._graph.vertices()}
            sigma = {v.vertex_id(): 0.0 for v in self._graph.vertices()}
            sigma[s] = 1.0
            d = {v.vertex_id(): -1 for v in self._graph.vertices()}
            d[s] = 0

            queue = deque([s])
            while queue:
                v = queue.popleft()
                S.append(v)
                v_obj = self._graph.get_vertex(v)
                for w_obj in self._graph._adjancencies[v_obj]:
                    w = w_obj.vertex_id()
                    # Caminho encontrado pela primeira vez
                    if d[w] < 0:
                        d[w] = d[v] + 1
                        queue.append(w)
                    # Caminho mínimo passa por v?
                    if d[w] == d[v] + 1:
                        sigma[w] += sigma[v]
                        P[w].append(v)

            # Passo 2: Acumulação de dependências
            delta = {v.vertex_id(): 0.0 for v in self._graph.vertices()}
            while S:
                w = S.pop()
                for v in P[w]:
                    delta[v] += (sigma[v] / sigma[w]) * (1.0 + delta[w])
                if w != s:
                    # Dividir por 2 porque o grafo não é orientado (arestas contadas 2x)
                    betweenness[w] += delta[w]

        # Ajuste final para grafos não orientados
        for v in betweenness:
            betweenness[v] /= 2.0
        return betweenness


    def _path(self, source, target):
        """Devolve a sequência de vértices do caminho mínimo source -> target (BFS)."""
        _, pred = self.bfs(source)
        if target not in pred:
            return None  # sem caminho
        path = []
        v = target
        while v is not None:
            path.append(v)
            v = pred[v]
        path.reverse()
        return path





# 1.1. Análise de complexidade
__init__(graph):


*  Complexidade: O(1) ou O(n+m)(se considerarmos o CSV)
*  Justificação: O construtor apenas guarda referências e consulta valores já calculados pelo objeto Graph. Se incluirmos a leitura do CSV, a inserção de n vértices e m arestas numa lista de adjacências demora O(n + m).

bfs(source):

*  Complexidade: O(n+m)
*  Justificação: Este método vai percorrer o grafo a partir de um determinado vértice, por isso, temos que considerar que cada vértice entra na fila uma única vez e partindo de cada um, vamos verificar as suas arestas.

num_components():


*   Complexidade: O(n+m)
*   Justificação: Para saber quantas componentes existem, o algoritmo não pode olhar apenas para o número total. Ele tem de percorrer o grafo todo. Primeiro, passa por cada um dos n vértices para ver se já foi visitado. Se encontrar um herói novo, faz um BFS para descobrir todos os amigos ligados a ele. No final, todas as m arestas e n heróis foram verificados exatamente uma vez para garantir que não ficou ninguém de fora. Por isso, o tempo que demora cresce de forma linear com o tamanho total do grafo (n+m).

largest_component():



*   Complexidade: O(n+m)
*   Justificação: Para devolver o grafo da maior componente, o algoritmo tem de primeiro identificar todas as "ilhas" de heróis. Para isso, é necessário percorrer todos os vértices (n) e verificar todas as arestas (m) através de um BFS. Só depois de ver as relações de todos é que conseguimos saber qual é o grupo com mais personagens e criar o novo grafo.

degree_distribution():


*   Complexidade: O(n).
*   Justificação: Percorremos a lista de n vértices uma vez e consultamos o seu grau (operação O(1)).


diameter():


*   Complexidade: O(n x (n + m)).
*   Justificação: O algoritmo executa um BFS completo a partir de cada um dos n vértices para encontrar a maior distância possível.


degree_centrality():


*   Complexidade: O(n).
*   Justificação: Um único ciclo sobre os n vértices para calcular a divisão do grau por (n-1).



closeness_centrality():


*   Complexidade: O(n x (n + m)).
*   Justificação: Para cada vértice, corremos um BFS para obter as distâncias a todos os outros nós.


eigenvector_centrality():


*   Complexidade: O(i x (n + m)).
*   Justificação: Sendo i o número de iterações, em cada iteração percorremos todos os vértices e os seus vizinhos para atualizar os pesos.



betweenness_centrality():


*   Complexidade: O(n x (n + m)).
*   Justificação: Utilizamos o Algoritmo de Brandes, que para grafos não pesados realiza um BFS modificado e uma fase de acumulação para cada um dos n vértices.


